# LangChain Price-Comparison Agent — Ollama Local LLM Backbone

This notebook walks through the full build from `PLAN.md` using a **local Ollama model** as the LLM backbone — completely free, no API key of any kind required.

- **Run A** — unmonitored baseline (custom execution-trace logger)
- **Run B** — AgentTrust-wrapped (per-step governance + final-output validation)
- **Analysis** — join the two traces on `correlation_id` and compare

**Prerequisites before running:**
1. `brew install ollama` (or download from ollama.com)
2. `ollama pull llama3.1`
3. `ollama serve` running in a terminal
4. Confirm: `curl http://localhost:11434/api/tags` shows `llama3.1`

## Step 0 — Environment Setup

Install all required packages. No `langchain-openai` or any API key package needed.
Run once; restart kernel if prompted.

In [1]:
import subprocess, sys

packages = [
    "langchain==0.3.9",
    "langchain-community==0.3.9",
    "langchain-ollama==0.2.3",   # pinned: latest 0.3.x needs langchain-core > 0.3.9
    "requests",
    "beautifulsoup4",
    "python-dotenv",
    "pandas",
    "matplotlib",
]

for pkg in packages:
    subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-q", pkg])

print("✓ Packages installed (no API key packages)")

# AgentTrust SDK — adjust path and uncomment:
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/path/to/agentrust_sdk"])
print("⚠  AgentTrust install line is commented out — edit the path above and uncomment")

✓ Packages installed (no API key packages)
⚠  AgentTrust install line is commented out — edit the path above and uncomment


In [3]:
import os
from pathlib import Path

for d in ["agent", "runs", "analysis", "policy", "logs"]:
    Path(d).mkdir(exist_ok=True)

print("Directory structure ready:")
for d in ["agent", "runs", "analysis", "policy", "logs"]:
    print(f"  ./{d}/")

Directory structure ready:
  ./agent/
  ./runs/
  ./analysis/
  ./policy/
  ./logs/


## Step 0b — Verify Ollama is Running

No API key needed. Confirm the local Ollama server is up and `llama3.1` is pulled.

In [ ]:
import requests as _req
from pathlib import Path
from dotenv import load_dotenv

# Check Ollama server
try:
    tags = _req.get("http://localhost:11434/api/tags", timeout=3).json()
    models = [m["name"] for m in tags.get("models", [])]
    print(f"✓ Ollama server running — models: {models}")
    if any("llama3.1" in m for m in models):
        print("✓ llama3.1 ready")
    else:
        print("⚠  llama3.1 not found — run: ollama pull llama3.1")
except Exception as e:
    print(f"✗ Ollama not reachable: {e}")
    print("  Start it with: ollama serve")

# Minimal .env — only AgentTrust vars needed (no LLM key)
env_content = """\
# LLM: Ollama runs locally on localhost:11434 — no API key required

# AgentTrust
AGENTRUST_KEY=at_...
AGENTRUST_GATEWAY_URL=http://localhost:8765
AGENTRUST_FAILURE_MODE=open
AGENTRUST_ENABLED=true
"""
env_path = Path(".env")
if not env_path.exists():
    env_path.write_text(env_content)
    print(".env created")
else:
    print(".env already exists — skipping overwrite")

load_dotenv(override=True)

## Step 1 — Define the Tools

Three standalone functions — test each one **before** wiring into LangChain. These are identical to `building-agent.ipynb`; tools are LLM-agnostic.

| Tool | What it does |
|---|---|
| `fetch_html` | GET an approved URL, return HTML + timing |
| `extract_price` | Parse a `$XX.XX` price out of HTML |
| `compare_prices` | Return the cheaper site + dollar difference |

In [4]:
%%writefile agent/tools.py
import re
import time
import requests
from bs4 import BeautifulSoup

# Replace with the two real product-page domains you want to compare
APPROVED_DOMAINS = ["site-a.example.com", "site-b.example.com"]


def fetch_html(url: str) -> dict:
    """Fetch raw HTML for an approved domain. Returns status, size, timing, html."""
    domain = url.split("/")[2]
    if domain not in APPROVED_DOMAINS:
        raise ValueError(f"Domain not approved: {domain}")

    start = time.time()
    resp = requests.get(url, timeout=10)
    elapsed = time.time() - start

    return {
        "url": url,
        "status": resp.status_code,
        "html": resp.text,
        "size": len(resp.text),
        "elapsed_sec": round(elapsed, 3),
    }


def extract_price(html: str) -> dict:
    """Parse a price out of HTML. Returns value + match count for validation."""
    soup = BeautifulSoup(html, "html.parser")
    matches = re.findall(r"\$\s?(\d+\.\d{2})", soup.get_text())
    return {
        "matches_found": len(matches),
        "price": float(matches[0]) if len(matches) == 1 else None,
        "raw_matches": matches,
    }


def compare_prices(price_a: float, price_b: float) -> dict:
    if price_a is None or price_b is None:
        return {"error": "Missing price — cannot compare"}
    cheaper = "A" if price_a < price_b else "B"
    return {
        "cheaper_site": cheaper,
        "price_a": price_a,
        "price_b": price_b,
        "difference": round(abs(price_a - price_b), 2),
    }

Overwriting agent/tools.py


### 1b — Smoke-test the tools with mock data

In [5]:
import sys
from unittest.mock import patch, MagicMock

if "agent.tools" in sys.modules:
    del sys.modules["agent.tools"]

from agent.tools import fetch_html, extract_price, compare_prices

mock_resp = MagicMock()
mock_resp.status_code = 200
mock_resp.text = '<html><body><span class="price">$29.99</span></body></html>'

with patch("agent.tools.requests.get", return_value=mock_resp):
    result_fetch = fetch_html("http://site-a.example.com/product")

print("fetch_html  →", {k: v for k, v in result_fetch.items() if k != "html"})

result_extract = extract_price(result_fetch["html"])
print("extract_price →", result_extract)
assert result_extract["price"] == 29.99

result_compare = compare_prices(29.99, 34.50)
print("compare_prices →", result_compare)
assert result_compare["cheaper_site"] == "A"

print("\n✓ All three tools pass smoke tests")

fetch_html  → {'url': 'http://site-a.example.com/product', 'status': 200, 'size': 59, 'elapsed_sec': 0.0}
extract_price → {'matches_found': 1, 'price': 29.99, 'raw_matches': ['29.99']}
compare_prices → {'cheaper_site': 'A', 'price_a': 29.99, 'price_b': 34.5, 'difference': 4.51}

✓ All three tools pass smoke tests


## Step 2 — Wrap as LangChain `StructuredTool`s

In [6]:
%%writefile agent/tools_wrapped.py
import json

from langchain.tools import StructuredTool
from pydantic import BaseModel
from agent.tools import fetch_html, extract_price, compare_prices


class FetchInput(BaseModel):
    url: str


class ExtractInput(BaseModel):
    html: str


class CompareInput(BaseModel):
    input: str


def _compare_prices_from_text(input: str) -> dict:
    """Adapter for the ReAct agent's single-string action input.
    Accepts a JSON object string or a 'price_a, price_b' pair."""
    try:
        data = json.loads(input)
        price_a, price_b = data["price_a"], data["price_b"]
    except (json.JSONDecodeError, KeyError, TypeError):
        parts = [p.strip().strip("$") for p in input.replace(",", " ").split()]
        price_a, price_b = parts[0], parts[1]
    return compare_prices(float(price_a), float(price_b))


fetch_tool = StructuredTool.from_function(
    func=fetch_html,
    name="fetch_html",
    description="Fetch HTML from an approved product page URL.",
    args_schema=FetchInput,
)

extract_tool = StructuredTool.from_function(
    func=extract_price,
    name="extract_price",
    description="Extract a numeric price from fetched HTML.",
    args_schema=ExtractInput,
)

compare_tool = StructuredTool.from_function(
    func=_compare_prices_from_text,
    name="compare_prices",
    description=(
        "Compare two prices and return the cheaper option. "
        'Input must be a single JSON object string, e.g. {"price_a": 49.99, "price_b": 44.49}'
    ),
    args_schema=CompareInput,
)

Overwriting agent/tools_wrapped.py


### 2b — Verify tool schemas

In [7]:
import json, sys

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.tools_wrapped import fetch_tool, extract_tool, compare_tool

for t in [fetch_tool, extract_tool, compare_tool]:
    schema = t.args_schema.model_json_schema()
    print(f"{t.name:20s} → {json.dumps(schema['properties'])}")

print("\n✓ All tool schemas look good")

fetch_html           → {"url": {"title": "Url", "type": "string"}}
extract_price        → {"html": {"title": "Html", "type": "string"}}
compare_prices       → {"input": {"title": "Input", "type": "string"}}

✓ All tool schemas look good


## Step 3 — Custom Execution-Trace Callback Handler

The `correlation_id` generated at `tool_start` is the join key between this trace and the AgentTrust governance trace in Step 7.

In [8]:
%%writefile agent/callback_handler.py
import json
import time
import uuid
from collections import defaultdict
from langchain.callbacks.base import BaseCallbackHandler


class ExecutionTraceHandler(BaseCallbackHandler):
    """Writes a JSONL execution trace. Every tool-start/end pair shares a
    correlation_id so the governance trace can be joined on that key."""

    def __init__(self, run_id: str, log_path: str = "logs/execution_trace.jsonl"):
        self.run_id = run_id
        self.log_path = log_path
        self._pending: dict[int, str] = {}
        self._call_counter = defaultdict(int)

    def correlation_id_for(self, tool_name: str) -> str:
        key = self._call_counter[tool_name] - 1
        return self._pending.get(key, str(uuid.uuid4()))

    def _write(self, record: dict):
        record["run_id"] = self.run_id
        record["timestamp"] = time.time()
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

    def on_tool_start(self, serialized, input_str, **kwargs):
        cid = str(uuid.uuid4())
        serial = self._call_counter[serialized.get("name", "")]
        self._call_counter[serialized.get("name", "")] += 1
        self._pending[serial] = cid
        self._write({
            "event": "tool_start",
            "tool": serialized.get("name"),
            "input": input_str,
            "correlation_id": cid,
        })

    def on_tool_end(self, output, **kwargs):
        tool_name = kwargs.get("name", list(self._call_counter.keys())[-1] if self._call_counter else "unknown")
        serial = self._call_counter[tool_name] - 1
        cid = self._pending.get(serial, str(uuid.uuid4()))
        self._write({
            "event": "tool_end",
            "tool": tool_name,
            "output": str(output),
            "correlation_id": cid,
        })

    def on_agent_action(self, action, **kwargs):
        self._write({
            "event": "agent_action",
            "tool": action.tool,
            "tool_input": action.tool_input,
            "log": action.log,
            "correlation_id": str(uuid.uuid4()),
        })

Overwriting agent/callback_handler.py


### 3b — Verify correlation_id pairing

In [9]:
import uuid, json, sys
from pathlib import Path

test_log = Path("logs/test_handler.jsonl")
test_log.unlink(missing_ok=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.callback_handler import ExecutionTraceHandler

handler = ExecutionTraceHandler(run_id=str(uuid.uuid4()), log_path=str(test_log))
handler.on_tool_start({"name": "fetch_html"}, '{"url": "http://site-a.example.com/product"}')
handler.on_tool_end('{"status": 200}', name="fetch_html")

records = [json.loads(l) for l in test_log.read_text().splitlines()]
for r in records:
    print(f"  event={r['event']:12s}  correlation_id={r['correlation_id']}")

assert records[0]["correlation_id"] == records[1]["correlation_id"]
print("\n✓ start/end share the same correlation_id")

  event=tool_start    correlation_id=997c6b20-789b-4e80-adde-ca50850ecdea
  event=tool_end      correlation_id=997c6b20-789b-4e80-adde-ca50850ecdea

✓ start/end share the same correlation_id


## Step 4 — Smoke-test OllamaLLM

Verify `langchain-ollama` can reach the local server before building the full agent.

In [2]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1")
response = llm.invoke("Reply with the single word: ok")
print(f"Ollama responded: {repr(response)}")
assert "ok" in response.lower(), f"Unexpected response: {response}"
print("✓ OllamaLLM round-trip works")

Ollama responded: 'ok'
✓ OllamaLLM round-trip works


## Step 4b — Build the Agent

In [1]:
%%writefile agent/build_agent.py
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from agent.tools_wrapped import fetch_tool, extract_tool, compare_tool
from agent.output_parser import CleanReActOutputParser

llm = OllamaLLM(model="llama3.1")

REACT_PROMPT = PromptTemplate.from_template(
    "You are an agent that compares product prices across two approved websites.\n\n"
    "Tools available: {tool_names}\n\n"
    "Tool descriptions:\n{tools}\n\n"
    "Format (follow exactly, one item per line, no backticks):\n"
    "Thought: <reasoning>\n"
    "Action: <tool name exactly as listed above>\n"
    "Action Input: <input>\n"
    "Observation: <tool result>\n\n"
    "STOP RULE: As soon as compare_prices returns an Observation with cheaper_site, "
    "price_a, price_b, and difference — stop calling tools. "
    "Output the raw JSON string from that Observation unchanged as the Final Answer.\n\n"
    "Begin!\n\n"
    "Question: {input}\n"
    "Thought:{agent_scratchpad}"
)

tools = [fetch_tool, extract_tool, compare_tool]
agent = create_react_agent(llm, tools, REACT_PROMPT, output_parser=CleanReActOutputParser())
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True,
    handle_parsing_errors=True, max_iterations=20,
)


Overwriting agent/build_agent.py


In [2]:
%%writefile agent/output_parser.py
import re
from langchain.agents.output_parsers import ReActSingleInputOutputParser


class CleanReActOutputParser(ReActSingleInputOutputParser):
    """Strips backticks/keyword prefixes that llama3.1 adds to Action lines."""

    def parse(self, text: str):
        # Action: `fetch_html`  ->  Action: fetch_html
        text = re.sub(r"^Action:\s*`([^`]+)`\s*$", r"Action: \1", text, flags=re.MULTILINE)
        # Action Input: `url="http://..."` -> Action Input: http://...
        text = re.sub(
            r'^Action Input:\s*`?(?:url=)?["\']?(https?://[^\s"\'`\)]+)["\']?`?\s*$',
            r"Action Input: \1", text, flags=re.MULTILINE,
        )
        # Action Input: `{...}`  ->  Action Input: {...}
        text = re.sub(r"^Action Input:\s*`(.+)`\s*$", r"Action Input: \1", text, flags=re.MULTILINE)
        return super().parse(text)

Overwriting agent/output_parser.py


### 4c — Import check (no LLM call yet)

In [3]:
import sys

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
print(f"AgentExecutor ready — tools: {[t.name for t in agent_executor.tools]}")
print("✓ Import check passed")

AgentExecutor ready — tools: ['fetch_html', 'extract_price', 'compare_prices']
✓ Import check passed


## Step 5 — Run A: Unmonitored Baseline

**Before running:**
- `ollama serve` must be running
- Replace `SITE_A_URL` and `SITE_B_URL` with real URLs that return HTML with a single `$XX.XX` price

Produces `logs/execution_trace.jsonl` — baseline with no governance overhead.

In [4]:
import sys, uuid, json
from pathlib import Path
from unittest.mock import patch, MagicMock
from dotenv import load_dotenv
load_dotenv(override=True)

for d in ["agent", "runs", "analysis", "policy", "logs"]:
    Path(d).mkdir(exist_ok=True)

Path("logs/execution_trace.jsonl").unlink(missing_ok=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
from agent.callback_handler import ExecutionTraceHandler

SITE_A_URL = "http://site-a.example.com/product"
SITE_B_URL = "http://site-b.example.com/product"
PRODUCT    = "Widget Pro 3000"

# Mock HTTP so the agent can complete without real URLs.
# Replace these with real pages when you have live product URLs.
def _mock_get(url, timeout=10):
    resp = MagicMock()
    resp.status_code = 200
    if "site-a" in url:
        resp.text = '<html><body><span class="price">$49.99</span></body></html>'
    else:
        resp.text = '<html><body><span class="price">$44.49</span></body></html>'
    return resp

run_id_a = str(uuid.uuid4())
handler_a = ExecutionTraceHandler(run_id=run_id_a)

with patch("agent.tools.requests.get", side_effect=_mock_get):
    result_a = agent_executor.invoke(
        {"input": f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}"},
        config={"callbacks": [handler_a]},
    )

print("\n=== Run A Result ===")
print(result_a)
print(f"\nRun ID : {run_id_a}")
print("Trace  : logs/execution_trace.jsonl")




> Entering new AgentExecutor chain...
We need to fetch the HTML content from both product pages, extract the prices from each page, and then compare those prices to determine which site has the cheaper option.

Action: fetch_html 
Action Input: http://site-a.example.com/product{'url': 'http://site-a.example.com/product', 'status': 200, 'html': '<html><body><span class="price">$49.99</span></body></html>', 'size': 59, 'elapsed_sec': 0.001}We've successfully fetched the HTML content from site-a's product page.

Action: extract_price
Action Input: {'url': 'http://site-a.example.com/product', 'status': 200, 'html': '<html><body><span class="price">$49.99</span></body></html>', 'size': 59, 'elapsed_sec': 0.001}{'matches_found': 1, 'price': 49.99, 'raw_matches': ['49.99']}Action: fetch_html 
Action Input: http://site-b.example.com/product{'url': 'http://site-b.example.com/product', 'status': 200, 'html': '<html><body><span class="price">$44.49</span></body></html>', 'size': 59, 'elapsed_se

### 5b — Inspect the execution trace

In [5]:
import pandas as pd, json
from pathlib import Path

trace_path = Path("logs/execution_trace.jsonl")
if trace_path.exists():
    df_exec = pd.DataFrame(
        [json.loads(l) for l in trace_path.read_text().splitlines() if l.strip()]
    )
    print(f"Total records: {len(df_exec)}")
    print(df_exec[["event", "tool", "correlation_id"]].to_string())
else:
    print("No trace yet — run Step 5 first")

Total records: 24
           event            tool                        correlation_id
0   agent_action      fetch_html  ccc93a53-b6c7-4082-859e-9b7f479a9323
1     tool_start      fetch_html  d9b96b4d-7e36-48fc-8556-ec25658d3d7b
2       tool_end      fetch_html  d9b96b4d-7e36-48fc-8556-ec25658d3d7b
3   agent_action   extract_price  bc50ad77-2b78-44e1-af36-6dd39f868408
4     tool_start   extract_price  e3327bbf-2196-436f-ac28-fcae23e1698b
5       tool_end   extract_price  e3327bbf-2196-436f-ac28-fcae23e1698b
6   agent_action      fetch_html  c9f37347-dfc6-4488-8bc0-772eb7b8648d
7     tool_start      fetch_html  914595ce-1f13-4ac2-b654-297b123bec84
8       tool_end      fetch_html  914595ce-1f13-4ac2-b654-297b123bec84
9   agent_action      fetch_html  9524d296-cb37-4de9-a539-e4f972a97504
10    tool_start      fetch_html  811949b9-4836-428e-8459-245e526abd90
11      tool_end      fetch_html  811949b9-4836-428e-8459-245e526abd90
12  agent_action   extract_price  a15918fd-c385-48dd-8b13-a

## Step 6 — AgentTrust Policy

Write the YAML policy file. The embedded gateway picks this up automatically.

In [3]:
policy_yaml = """\
meta:
  id: price-agent
  version: "1.0"
  description: "Rules for the LangChain price-comparison agent (Claude CLI backbone)"
  patterns:
    - "price-comparison-agent"

rules:
  - id: output_cheaper_site_present
    description: "Output must include a cheaper_site field"
    severity: high
    target: output.cheaper_site
    op: exists
    effect: deny

  - id: output_price_a_present
    description: "Output must include price_a"
    severity: high
    target: output.price_a
    op: exists
    effect: deny

  - id: output_price_b_present
    description: "Output must include price_b"
    severity: high
    target: output.price_b
    op: exists
    effect: deny

  - id: approved_domains_only
    description: "Only approved domains may be fetched"
    severity: critical
    target: output.domains_used
    op: not_in_list
    value: ["evil.com", "shadow-site.net"]
    effect: deny

  - id: large_price_difference_review
    description: "Differences above $50 are escalated for human review"
    severity: medium
    target: output.difference
    op: lte
    value: 50.0
    effect: review
"""

from pathlib import Path
policy_path = Path("policy/price_agent.yaml")
policy_path.write_text(policy_yaml)
print(f"✓ Policy written to {policy_path}")

✓ Policy written to policy/price_agent.yaml


## Step 7 — Run B: AgentTrust-Wrapped

Uses **Pattern C** (direct `client.validate()`) for per-step governance with full `ValidateResponse` access.

**Requires:** AgentTrust SDK installed (Step 0) and `AGENTRUST_KEY` set in `.env`.

In [6]:
import sys, uuid, json, time
from pathlib import Path
from unittest.mock import patch, MagicMock
from dotenv import load_dotenv
load_dotenv(override=True)

Path("logs").mkdir(exist_ok=True)
Path("logs/governance_trace.jsonl").unlink(missing_ok=True)

SITE_A_URL = "http://site-a.example.com/product"
SITE_B_URL = "http://site-b.example.com/product"
PRODUCT    = "Widget Pro 3000"

from agentrust_sdk import embed_gateway, AgentTrustClient, BlockedError

gw     = embed_gateway()
client = AgentTrustClient()

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
from agent.callback_handler import ExecutionTraceHandler

AGENT_ID = "price-comparison-agent"
USER     = "qa-runner"
run_id_b = str(uuid.uuid4())
handler_b = ExecutionTraceHandler(run_id=run_id_b)

_orig_on_tool_end = handler_b.on_tool_end

def _governed_on_tool_end(output, **kwargs):
    tool_name = kwargs.get("name", "unknown")
    serial = handler_b._call_counter.get(tool_name, 1) - 1
    cid    = handler_b._pending.get(serial, str(uuid.uuid4()))

    blocked = False
    record = {
        "event": "tool_governance", "tool": tool_name,
        "correlation_id": cid, "run_id": run_id_b, "timestamp": time.time(),
    }

    try:
        try:
            tool_output = json.loads(output) if isinstance(output, str) else output
        except Exception:
            tool_output = {"raw": str(output)}

        r = client.validate(
            agent_id=AGENT_ID, user=USER,
            input=f"tool_call:{tool_name}", output=tool_output,
            framework="LangChain",
            metadata={"correlation_id": cid, "run_id": run_id_b},
        )
        try:
            record.update({
                "envelope_id":      r.envelope_id,
                "outcome":          r.decision.outcome,
                "decision_reason":  r.decision.reason,
                "policy_version":   r.decision.policy_version,
                "risk_tier":        r.risk.tier,
                "risk_score":       r.risk.score,
                "risk_reason":      r.risk.reason,
                "policy_score":     r.validation.policy_score,
                "final_confidence": r.validation.final_confidence,
                "failures":         r.validation.failures,
            })
        except Exception as e:
            record["sdk_error"] = str(e)[:120]

        if getattr(r, "blocked", False):
            blocked = True

    except Exception as e:
        record["outcome"] = "sdk_error"
        record["sdk_error"] = str(e)[:120]

    with open("logs/governance_trace.jsonl", "a") as f:
        f.write(json.dumps(record, default=str) + "\n")

    if blocked:
        raise BlockedError(reason=r.decision.reason, envelope_id=r.envelope_id)

    _orig_on_tool_end(output, **kwargs)

handler_b.on_tool_end = _governed_on_tool_end

def _mock_get(url, timeout=10):
    resp = MagicMock()
    resp.status_code = 200
    resp.text = (
        '<html><body><span class="price">$49.99</span></body></html>'
        if "site-a" in url else
        '<html><body><span class="price">$44.49</span></body></html>'
    )
    return resp

try:
    with patch("agent.tools.requests.get", side_effect=_mock_get):
        result_b = agent_executor.invoke(
            {"input": f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}"},
            config={"callbacks": [handler_b]},
        )
except BlockedError as e:
    print(f"BLOCKED mid-execution: {e.reason}  (envelope_id={e.envelope_id})")
    result_b = None

if result_b:
    final_output = result_b.get("output", result_b)
    if isinstance(final_output, str):
        try:
            final_output = json.loads(final_output)
        except Exception:
            final_output = {"answer": final_output}

    gov_record = {
        "event": "final_governance", "correlation_id": "final",
        "run_id": run_id_b, "timestamp": time.time(),
    }
    try:
        r_final = client.validate(
            agent_id=AGENT_ID, user=USER,
            input=f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}",
            output=final_output, framework="LangChain",
            metadata={"step": "final_output", "run_id": run_id_b},
        )
        try:
            gov_record.update({
                "envelope_id":      r_final.envelope_id,
                "outcome":          r_final.decision.outcome,
                "decision_reason":  r_final.decision.reason,
                "risk_tier":        r_final.risk.tier,
                "policy_score":     r_final.validation.policy_score,
                "final_confidence": r_final.validation.final_confidence,
                "failures":         r_final.validation.failures,
            })
        except Exception as e:
            gov_record["sdk_error"] = str(e)[:120]
    except Exception as e:
        gov_record["outcome"] = "sdk_error"
        gov_record["sdk_error"] = str(e)[:120]

    with open("logs/governance_trace.jsonl", "a") as f:
        f.write(json.dumps(gov_record, default=str) + "\n")

    print(f"Run B complete — output: {final_output}")

gw.stop()
print(f"\nRun ID : {run_id_b}")
print("Governance trace → logs/governance_trace.jsonl")




> Entering new AgentExecutor chain...


[AgentTrust] Gateway unreachable (fail-open): Client error '401 Unauthorized' for url 'http://127.0.0.1:8765/v1/runtime/validate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


We need to fetch the HTML from both product pages, extract the prices from each HTML, and then compare those prices to determine which site is cheaper.

Action: fetch_html
Action Input: http://site-a.example.com/product{'url': 'http://site-a.example.com/product', 'status': 200, 'html': '<html><body><span class="price">$49.99</span></body></html>', 'size': 59, 'elapsed_sec': 0.0}

[AgentTrust] Gateway unreachable (fail-open): Client error '401 Unauthorized' for url 'http://127.0.0.1:8765/v1/runtime/validate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Action: extract_price
Action Input: <html><body><span class="price">$49.99</span></body></html>{'matches_found': 1, 'price': 49.99, 'raw_matches': ['49.99']}

...

Action: extract_price
Action Input: <html><body><span class="price">$44.49</span></body></html>{'matches_found': 1, 'price': 44.49, 'raw_matches': ['44.49']}

[AgentTrust] Gateway unreachable (fail-open): Client error '401 Unauthorized' for url 'http://127.0.0.1:8765/v1/runtime/validate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Here is the continuation of your thought process:

Action: compare_prices
Action Input: {"price_a": 49.99, "price_b": 44.49}
{'cheaper_site': 'B', 'price_a': 49.99, 'price_b': 44.49, 'difference': 5.5}

[AgentTrust] Gateway unreachable (fail-open): Client error '401 Unauthorized' for url 'http://127.0.0.1:8765/v1/runtime/validate'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Since compare_prices has already been called and returned an Observation with the cheaper site, price A, price B, and difference, we should stop calling tools now.

Final Answer:

{'cheaper_site': 'B', 'price_a': 49.99, 'price_b': 44.49, 'difference': 5.5}

> Finished chain.
Run B complete — output: {'answer': "{'cheaper_site': 'B', 'price_a': 49.99, 'price_b': 44.49, 'difference': 5.5}"}

Run ID : 8ff049f7-79ec-44f8-9edd-7fa352ba7251
Governance trace → logs/governance_trace.jsonl


## Step 8 — Join and Compare Traces

Join the two JSONL logs on `correlation_id`.
Run A rows have no governance columns; Run B rows carry `outcome`, `risk_tier`, `policy_score`, and `failures`.

In [7]:
import pandas as pd, json
from pathlib import Path

def load_jsonl(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    return pd.DataFrame([json.loads(l) for l in p.read_text().splitlines() if l.strip()])

exec_trace = load_jsonl("logs/execution_trace.jsonl")
gov_trace  = load_jsonl("logs/governance_trace.jsonl")

if exec_trace.empty:
    print("No execution trace — run Step 5 first")
elif gov_trace.empty:
    print("No governance trace — run Step 7 first")
else:
    merged = exec_trace.merge(gov_trace, on="correlation_id", how="left", suffixes=("_exec", "_gov"))
    Path("analysis").mkdir(exist_ok=True)
    merged.to_csv("analysis/merged_trace.csv", index=False)

    print("=== Governance Outcome Summary ===")
    if "outcome" in merged.columns:
        print(merged.groupby("outcome")["correlation_id"].count().rename("calls"))

    print("\n=== Blocked or Escalated Steps ===")
    if "outcome" in merged.columns:
        flagged = merged[merged["outcome"].isin(["block", "escalate"])]
        cols = [c for c in ["tool_exec", "outcome", "decision_reason", "risk_tier", "policy_score", "failures"] if c in flagged.columns]
        print(flagged[cols].to_string() if not flagged.empty else "None")

    print("\n=== Risk Tier Distribution ===")
    if "risk_tier" in merged.columns and "policy_score" in merged.columns:
        print(merged.groupby("risk_tier")["policy_score"].agg(["count", "mean"]))

    print("\nMerged CSV → analysis/merged_trace.csv")

=== Governance Outcome Summary ===
outcome
sdk_error    10
Name: calls, dtype: int64

=== Blocked or Escalated Steps ===
None

=== Risk Tier Distribution ===

Merged CSV → analysis/merged_trace.csv


### 8b — What governance caught that Run A missed

In [8]:
import pandas as pd
from pathlib import Path

csv_path = Path("analysis/merged_trace.csv")
if not csv_path.exists():
    print("Run Step 8 first to generate the merged CSV")
else:
    df = pd.read_csv(csv_path)
    flagged_mask = df["outcome"].isin(["block", "escalate"]) if "outcome" in df.columns else pd.Series(False, index=df.index)
    flagged = df[flagged_mask]

    if flagged.empty:
        print("No blocks or escalations — all tool outputs passed governance in Run B")
    else:
        print(f"Governance caught {len(flagged)} flagged step(s) that Run A did not:")
        for _, row in flagged.iterrows():
            print(f"\n  tool          : {row.get('tool_exec', row.get('tool', '?'))}")
            print(f"  outcome       : {row.get('outcome')}")
            print(f"  reason        : {row.get('decision_reason')}")
            print(f"  risk_tier     : {row.get('risk_tier')}")
            print(f"  policy_score  : {row.get('policy_score')}")
            print(f"  failures      : {row.get('failures')}")

No blocks or escalations — all tool outputs passed governance in Run B
